# Learning rate finder a Mamba projekthez

Ez a notebook a `src/random_seed.py`, `src/LR_finder.py` és `src/LR_finder_backbone.py` működését mutatja be kommentekkel. A projekt két külön keresést használ:

1. **Warm-up LR finder:** frissen létrehozott ResNet50 transfer-learning modellhez keres kezdeti tanulási rátát.
2. **Backbone LR finder:** egy már bemelegített modell felső ResNet rétegeinek kiolvasztása után keres finomhangolási tanulási rátát.

A drága, 200 lépéses tanítási cellák szándékosan ki vannak kapcsolva. Ezeket csak akkor futtasd, ha az adatok, a képek, az ImageNet-súlyok és a szükséges TensorFlow-környezet rendelkezésre áll.

## 1. Projektútvonal és determinisztikus seed

A notebook gyökerét a fájl helyzetéből számoljuk ki, így a relatív `src/` és `data/` útvonalak akkor is működnek, ha a notebookot a Jupyterből nyitjuk meg. A `set_global_seed` a Python, NumPy, TensorFlow és Keras véletlenszám-generátorait állítja be. Ez nem garantál minden GPU-műveletre tökéletes determinizmust; ehhez a forrásban szereplő opcionális op-determinizmust is be kell kapcsolni.

In [ ]:
from pathlib import Path
import os
import sys

# A notebook a repository/notebooks mappában van, ezért a projektgyökér a szülőmappa.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# A forrásfájlok egymás közötti importjai (például: from random_seed import ...) miatt kell.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from random_seed import set_global_seed

SEED = 42
set_global_seed(SEED)
print(f"Projektgyökér: {PROJECT_ROOT}")
print(f"Seed beállítva: {SEED}")

## 2. TensorFlow-környezet ellenőrzése

A két LR finder mixed-precision módot állít be a forrásfájlban. A következő cella csak ellenőriz; nem indít tanítást.

In [ ]:
import tensorflow as tf

print("TensorFlow verzió:", tf.__version__)
print("Látható GPU-k:", tf.config.list_physical_devices("GPU"))
print("Jelenlegi Keras policy:", tf.keras.mixed_precision.global_policy().name)

## 3. A forrásimplementációk újrahasználata

Nem másoljuk be a callback és a modellek logikáját a notebookba. Az implementációk továbbra is az `src` mappában maradnak, így a notebook ugyanazt a kódot használja, mint a parancssorból futtatott programok.

- `LR_finder.LRFinder`: warm-up keresés, alapértelmezett tartomány `1e-6` és `1e-1` között.
- `LR_finder_backbone.LRFinder`: backbone-finetuning keresés, alapértelmezett tartomány `1e-7` és `1e-2` között.
- Mindkettő exponenciálisan növeli a learning rate-et, simított loss-t gyűjt, majd a loss legmeredekebb eséséből javaslatot számol.

Az import a modulok `if __name__ == "__main__"` blokkját nem futtatja le, ezért itt nem indul el automatikus tréning.

In [ ]:
import importlib

lr_finder_module = importlib.import_module("LR_finder")
backbone_module = importlib.import_module("LR_finder_backbone")

WarmupLRFinder = lr_finder_module.LRFinder
BackboneLRFinder = backbone_module.LRFinder

print("Warm-up callback:", WarmupLRFinder)
print("Backbone callback:", BackboneLRFinder)
print("Warm-up alapértelmezett LR:", lr_finder_module.DEFAULT_WARMUP_LR)
print("Backbone alapértelmezett LR:", backbone_module.DEFAULT_FINETUNE_LR)

## 4. Adatok és képek előzetes ellenőrzése

A dataset ugyanazt a `train_split_fixed.csv` fájlt és a `train_images_224_letterboxed` könyvtárat használja, mint a forrás. A cella hibát jelez, ha egy szükséges erőforrás hiányzik; a tényleges képfeldolgozás csak a későbbi, opcionális futtatáskor történik.

In [ ]:
import pandas as pd

TRAIN_CSV = DATA_DIR / "train_split_fixed.csv"
IMAGE_DIR = DATA_DIR / "train_images_224_letterboxed"

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Hiányzik a tanító CSV: {TRAIN_CSV}")
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Hiányzik a képkönyvtár: {IMAGE_DIR}")

train_df = pd.read_csv(TRAIN_CSV)
required_columns = {"image_path", "class_id", "MIVS"}
missing_columns = required_columns.difference(train_df.columns)
if missing_columns:
    raise ValueError(f"Hiányzó oszlop(ok): {sorted(missing_columns)}")

print("Tanítópéldák száma:", len(train_df))
print("Osztályok száma:", train_df["class_id"].nunique())
print("MIVS eloszlás:")
display(train_df["MIVS"].value_counts().sort_index())
print("Első képutak:")
display(train_df[["image_path", "class_id", "MIVS"]].head())

## 5. Warm-up modell és LR finder

A `LR_finder.py` modellje uint8 képet fogad, GPU-n augmentál, ResNet50 ImageNet-előfeldolgozást használ, majd két fejet ad:

- `species_head`: 296 faj softmax kimenete;
- `venomous_head`: egy sigmoid kimenet a mérgezőséghez.

A ResNet50 backbone ebben a szakaszban fagyasztott. A callback `1e-6`-ról `1e-1`-ig növeli a learning rate-et, ezért a keresés csak rövid, diagnosztikai futtatás.

In [ ]:
BATCH_SIZE = 128

# A forrásban lévő adatpipeline és modell használata biztosítja a viselkedés egyezését.
warmup_dataset = lr_finder_module.create_dataset(
    TRAIN_CSV, IMAGE_DIR, BATCH_SIZE, is_training=True
)
warmup_model, warmup_backbone = lr_finder_module.build_model(num_species=296)

compile_kwargs = {
    "loss": {
        "species_head": lr_finder_module.cost_sensitive_species_loss,
        "venomous_head": lr_finder_module.weighted_binary_venomous_loss,
    },
    "loss_weights": {"species_head": 1.0, "venomous_head": 0.2},
    "metrics": {
        "species_head": ["accuracy", lr_finder_module.venomous_penalty_metric],
        "venomous_head": "accuracy",
    },
    "jit_compile": False,
}

warmup_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    **compile_kwargs,
)
print("Warm-up modell elkészült.")
print("Fagyasztott backbone:", not warmup_backbone.trainable)
warmup_model.summary(show_trainable=True)

### Warm-up keresés futtatása (opcionális)

A cella alapból nem fut le automatikusan. A 200 lépés lefuttatása után a callbackből kiolvasható a javasolt `WARMUP_LR`, amelyet a `src/model.py` megfelelő konstansába lehet átvezetni.

In [ ]:
# Opcionális, költséges művelet: 200 batch tanítása növekvő learning rate mellett.
# warmup_finder = WarmupLRFinder(start_lr=1e-6, end_lr=1e-1, max_steps=200)
# warmup_model.fit(
#     warmup_dataset,
#     epochs=1,
#     steps_per_epoch=200,
#     callbacks=[warmup_finder],
#     verbose=1,
# )
# chosen_warmup_lr = warmup_finder.get_optimal_lr(
#     default_lr=lr_finder_module.DEFAULT_WARMUP_LR
# )
# print(f"Javasolt warm-up LR: {chosen_warmup_lr:.2e}")
# warmup_finder.plot_and_save(
#     MODELS_DIR / "lr_finder_plot.png",
#     chosen_warmup_lr,
# )

## 6. Backbone-finetuning LR finder

A `LR_finder_backbone.py` a már elmentett warm-up modellt tölti be, majd a ResNet50 utolsó rétegeit részben kiolvasztja. A `unfreeze_resnet_top` alapértelmezésben az utolsó 30 réteget teszi taníthatóvá, de a BatchNormalization rétegeket fagyasztva hagyja. Ezután a callback szűkebb, backbone-hoz illeszkedő `1e-7`–`1e-2` tartományban keres, és az eredményt `1e-6`–`1e-3` közé korlátozza.

In [ ]:
# A teljes backbone-kereséshez előbb a model.py vagy a warm-up folyamat által
# létrehozott models/snake_multitask_model.keras fájl szükséges.
MODEL_PATH = MODELS_DIR / "snake_multitask_model.keras"

if MODEL_PATH.exists():
    custom_objects = {
        "venomous_penalty_metric": backbone_module.venomous_penalty_metric,
        "cost_sensitive_species_loss": backbone_module.cost_sensitive_species_loss,
        "weighted_binary_venomous_loss": backbone_module.weighted_binary_venomous_loss,
    }
    with tf.keras.utils.custom_object_scope(custom_objects):
        finetune_model = tf.keras.models.load_model(str(MODEL_PATH))

    try:
        finetune_backbone = finetune_model.get_layer("resnet50")
    except ValueError as error:
        raise ValueError("A betöltött modellben nem található resnet50 réteg.") from error

    backbone_module.unfreeze_resnet_top(finetune_backbone, num_layers=30)
    finetune_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-7),
        **{
            "loss": {
                "species_head": backbone_module.cost_sensitive_species_loss,
                "venomous_head": backbone_module.weighted_binary_venomous_loss,
            },
            "loss_weights": {"species_head": 1.0, "venomous_head": 0.2},
            "metrics": {
                "species_head": ["accuracy", backbone_module.venomous_penalty_metric],
                "venomous_head": "accuracy",
            },
            "jit_compile": False,
        },
    )
    print("Finomhangolási modell betöltve és újrafordítva.")
    print("Tanítható ResNet rétegek:", sum(layer.trainable for layer in finetune_backbone.layers))
else:
    print(f"A backbone keresés kihagyva: nincs még itt a modell: {MODEL_PATH}")

### Backbone keresés futtatása (opcionális)

A javasolt értéket a `src/model.py` `FINETUNE_LR` konstansában lehet használni. A callback egy már tanított modell súlyait módosítja, ezért a keresés után érdemes a modellt újra betölteni vagy újra létrehozni a tényleges finomhangoláshoz.

In [ ]:
# Opcionális, költséges művelet: csak akkor aktiváld, ha finetune_model létrejött.
# finetune_dataset = backbone_module.create_dataset(
#     TRAIN_CSV, IMAGE_DIR, BATCH_SIZE, is_training=True
# )
# backbone_finder = BackboneLRFinder(start_lr=1e-7, end_lr=1e-2, max_steps=200)
# finetune_model.fit(
#     finetune_dataset,
#     epochs=1,
#     steps_per_epoch=200,
#     callbacks=[backbone_finder],
#     verbose=1,
# )
# chosen_finetune_lr = backbone_finder.get_optimal_lr(
#     default_lr=backbone_module.DEFAULT_FINETUNE_LR
# )
# print(f"Javasolt backbone finetune LR: {chosen_finetune_lr:.2e}")
# backbone_finder.plot_and_save(
#     MODELS_DIR / "lr_finder_backbone_plot.png",
#     chosen_finetune_lr,
# )

## 7. A callback működésének rövid összefoglalása

A `LRFinder` minden batch elején az előző learning rate-et megszorozza egy konstans faktorral. A batch végén elmenti a nyers és az exponenciálisan simított losst. Ha a loss NaN lesz vagy a legjobb loss többszörösére ugrik, a callback leállítja a keresést. A `get_optimal_lr` a zajos kezdetet és végét levágja, a loss gradiensének minimumát keresi, majd biztonsági tartományba szorítja az eredményt.

Fontos: az LR finder javaslat, nem bizonyított optimum. A kiválasztott értéket külön validációs futással kell ellenőrizni.